<a href="https://colab.research.google.com/github/EtzionR/LM4GeoAI/blob/main/Solutions/SOL_2_Text_to_Geo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Solution for Ex 2: **Text 2 Geo Data**
### created by Etzion Harari | Geo-AI Course

[**https://github.com/EtzionR/LM4GeoAI**](https://github.com/EtzionR/LM4GeoAI)

## Imports

In [1]:
from transformers import pipeline, AutoModelForTokenClassification, AutoTokenizer
from geopy.geocoders import Nominatim
from time import sleep as wait
from tqdm import tqdm

import networkx as nx
import pandas as pd
import folium

## Clone git Repo
[https://github.com/EtzionR/NLP4GeoAI](https://github.com/EtzionR/NLP4GeoAI)

In [2]:
%%bash
rm -rf NLP4GeoAI
git clone https://github.com/EtzionR/NLP4GeoAI.git

Cloning into 'NLP4GeoAI'...


## Load Data
The dataset created using the following code: [**preprocess_datasets.ipynb**](https://colab.research.google.com/github/EtzionR/LM4GeoAI/blob/main/Data/preprocess_datasets.ipynb)

In [3]:
df = pd.read_csv('NLP4GeoAI/Data/data.csv')

print(f'Dataframe shape: {df.shape}')

df.head()

Dataframe shape: (23072, 2)


,Text,Source
0,"Last week, Sen. Malcolm Wallop -LRB- R., Wyo. ...",ontonotes5
1,Rules that set standards for products or gover...,ontonotes5
2,Determining when handicapped access is require...,ontonotes5
3,"``It's very costly and time-consuming ,'' says...",ontonotes5
4,"Next to medical insurance, ``costs of complian...",ontonotes5


# --------------------------------------------------------------------

## Q1

#### A) Create a Hugging Face pipeline for NER with average aggregation_strategy enabled. Please use [dslim/distilbert-NER](https://huggingface.co/dslim/distilbert-NER) model.

#### B) Apply the pipeline to a sample sentence containing people, organizations, and locations.

#### C) Display the NER output in a pandas DataFrame.

In [4]:
MODEL = "dslim/bert-base-NER"
TEXT = "Sam Altman visited OpenAI in San Francisco."

ner = pipeline("ner",
               model=MODEL,
               aggregation_strategy='average')

output = ner(TEXT)

print('\n\n\nNER output:\n\n')

pd.DataFrame(output)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenCla




NER output:




,entity_group,score,word,start,end
0,PER,0.999328,Sam Altman,0,10
1,ORG,0.664299,OpenAI,19,25
2,LOC,0.999392,San Francisco,29,42


## Q2

#### A) Select the 10% first rows from the DataFrame.

#### B) Apply the NER model on each of the selected texts.

#### C) For every detected entity, attach metadata such as the **original text** and its **source**.

#### D) Aggregate all entity-level outputs into a single pandas DataFrame and display its shape.

In [ ]:
P = .1 # Run NER only on P% of df entries

outputs = []

for text, source in tqdm(df[:int(len(df)*P)].values):
    entites = ner(text)

    for entity in entites:

        entity['source'] = source
        entity['text'] = text

        outputs.append(entity)

outputs = pd.DataFrame(outputs)

print(f'\n\nNER output shape: {outputs.shape}\n')

outputs

 11%|█         | 247/2307 [00:08<00:47, 42.93it/s]

## Q3

#### 1) Filter the NER output DataFrame to include only location entities.

#### 2) Count the total number of location entity occurrences.

#### 3) Compute the top-10 most frequently occurring location names.

#### 4) Calculate the total number of mentions among the top-K entities and their percentage relative to all location entities.

#### 5) Display the resulting summary table.

In [ ]:
number_of_loc_entities = (outputs.entity_group=='LOC').sum()
print(f'Number of location (LOC) entities: {number_of_loc_entities}')

k = 10
top_k_locations = outputs[outputs.entity_group=='LOC'].word.value_counts().head(k).reset_index()
number_of_mentions = top_k_locations['count'].sum()
print(f'Number of mentions of the top-K entities: {number_of_mentions} ({round(number_of_mentions/number_of_loc_entities*100,1)}%)\n')

top_k_locations

## Q4

#### 1) Initialize a Nominatim geocoder with a custom user agent and timeout.

#### 2) Geocode the example place name into a geographic location.

#### 3) Display the output geocoding result, include the X (longtitue) and Y (latitude) coordinates

In [ ]:
example = "Tel Aviv, Israel"

geolocator = Nominatim(user_agent="GeoAI_Course_Geocoder", timeout=10)

location = geolocator.geocode(example)

print(f'Location full adress:\n{location.address}\n')
print(f'WGS84 GEO X = {round(location.longitude,6)}, Y = {round(location.latitude,6)}')

## Q5

#### Geocode each place name in the top 10 place name dataframe. Add to each entry in the dataframe its coordinates. Please define a delay interval of 1. seconds (at least) between each call to Nominatim.


In [ ]:
time_gap = 1.25

x_coords = []
y_coords = []

for placename in tqdm(top_k_locations.word):
    loc = geolocator.geocode(placename)

    x_coords.append(loc.longitude)
    y_coords.append(loc.latitude)

    wait(time_gap)

top_k_locations['x'] = x_coords
top_k_locations['y'] = y_coords

top_k_locations

## Q6

#### Create folium map of the Top 10 places in the Corpus

In [ ]:

fmap = folium.Map(location=[0, 0], zoom_start=3)

places = []
place_to_xy = {}

for name,x,y in zip(top_k_locations.word, top_k_locations.x, top_k_locations.y):
    folium.Marker([y,x], popup=name, tooltip=name).add_to(fmap)
    places.append(name)
    place_to_xy[name] = (y, x)

fmap

## Q7

#### Construct an undirected graph where nodes represent the top-10 location entities and edges indicate that two locations co-occur in the **same text**. Track edge weights based on the number of co-occurrences and report the final number of nodes and edges.

In [ ]:
topk_places = set(places)

edge_weight = {}

G = nx.Graph()

sub = outputs[['text', 'entity_group', 'word']]
sub['merged'] = [(entity, typ) for _, typ, entity in sub.values]

sub = pd.pivot_table(sub[['text', 'merged']], index='text', aggfunc=set)

for entity_set in sub.merged[sub.merged.str.len()>1]:

    entity_list = [*entity_set]

    for i in range(len(entity_list)):
        for j in range(i+1, len(entity_list)):
            placei = entity_list[i][0]
            placej = entity_list[j][0]
            if placei in topk_places and placej in topk_places:
                G.add_edge(placei,
                           placej)
                edge_weight[(placei, placej)] = edge_weight.get((placei, placej), 0) + 1


print(f'\n\n\nConnections Graph created!\n|V| = {len(G.nodes)}\n|E| = {len(G.edges)}\n\n')

## Q8

#### Display the constructed graph with folium. Draw each edge in the graph as polyline between the two places it connect.

In [ ]:
fmap = folium.Map(location=[0, 0], zoom_start=3)

for name,x,y in zip(top_k_locations.word, top_k_locations.x, top_k_locations.y):
    folium.Marker([y,x], popup=name, tooltip=name,icon = folium.Icon(color="blue") ).add_to(fmap)

for i,j in G.edges:
    if (i,j) in edge_weight:
        folium.PolyLine(locations=[place_to_xy[j],
                                   place_to_xy[i]],
                        color="blue",
                        opacity=0.3,
                        tooltip=f'Side 1: {i}<br>Side 2: {j}<br>Connections: {edge_weight[(i,j)]}',
                        weight=edge_weight[(i,j)]).add_to(fmap)

fmap

## Q - Bonus

#### 1) Identify the top three Person (PER) entities based on their frequency of occurrence in the corpus.

#### 2) Select from the output entities dataframe only the texts that mention at least one of these top three PER entities.

#### 3) For each Location (LOC) entity appearing in the selected texts, retrieve its geographic coordinates using the Nominatim geocoding service.

#### 4) Visualize the results on a Folium map, ensuring that all locations co-occurring with the same person are displayed using the same color.

Create by Etzion Harari | Geo-AI Course | [https://github.com/EtzionR/LM4GeoAI](https://github.com/EtzionR/LM4GeoAI)